# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 2: Refresh / Content Opportunity Scoring

ML Task Type: Ranking and Scoring (driven by a probabilistic Binary Classification backend).

Why this task?
While we will train the model to output a binary probability of decline, the operational problem we are solving is a priority bottleneck. Editors do not have the bandwidth to review pages blindly or look at an unordered list of "yes/no" classifications. They need a sorted queue. By framing this as a scoring and ranking task, we can sort our content inventory so that the absolute highest-risk, highest-impact pages bubble up to the top slots.

In [12]:
import pandas as pd

# Load the starter data to prove we are setting up our target lane slice
df = pd.read_csv("content_refresh_anonymized.csv")

# Quick sanity check on the distribution of our candidate pool
total_pages = len(df)
print("--- TASK INVENTORY SANITY CHECK ---")
print(f"Total pages in pool: {total_pages:,}")


--- TASK INVENTORY SANITY CHECK ---
Total pages in pool: 30,000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The Target: We will define "is_declining_label" as our target variable.

Where the label comes from:In this starter playground, our proxy target is derived from the observed performance trend:

is_declining_label = (trend_direction == "down")

Leakage Prevention Rule:To ensure absolute data integrity, we must respect the Label Trap. Because is_declining_label is constructed from trend_direction, both trend_direction and trend_pct must be strictly omitted as features during modeling. Including them would result in catastrophic circular leakage where the model simply memorizes the target definition instead of finding organic signals.

In [13]:
# Compute the exact label distribution in our dataset
declining_count = len(df[df['trend_direction'] == 'down'])
pct_declining = (declining_count / len(df)) * 100

print("--- TARGET DISTRIBUTION ---")
print(f"True Positive Class (is_declining_label = 1): {declining_count:,} ({pct_declining:.1f}%)")
print(f"Negative Class (is_declining_label = 0): {len(df) - declining_count:,} ({100 - pct_declining:.1f}%)")


--- TARGET DISTRIBUTION ---
True Positive Class (is_declining_label = 1): 16,262 (54.2%)
Negative Class (is_declining_label = 0): 13,738 (45.8%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The Success Metric: Precision@50 and Average Precision (AP).

What number means 'good'?

In a prioritization queue, we care most about the quality of our top recommendations.The baseline rules in the starter dataset only achieve a Precision@50 of 0.240 (meaning only 12 out of the top 50 suggested pages are actually declining).

A random forest model on the same data achieves a Precision@50 of 0.740 (37 out of 50 correct).

Therefore, we define a "good" target threshold for our work as achieving a Precision@50 of >= 0.70 (70% or more editorial hours utilized effectively) while outperforming our heuristic baseline model.

In [14]:
# Showing the target baseline we are working against
baseline_precision_at_50 = 0.240
target_model_precision_at_50 = 0.700

print("--- PERFORMANCE TARGETS ---")
print(f"Starter Heuristic Baseline P@50: {baseline_precision_at_50:.3f}")
print(f"Minimum Acceptable Model Target P@50: {target_model_precision_at_50:.3f}")


--- PERFORMANCE TARGETS ---
Starter Heuristic Baseline P@50: 0.240
Minimum Acceptable Model Target P@50: 0.700


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The Unit of Analysis: One single row represents one unique, pseudonymized content item (one page) belonging to a specific client.

This grain is verified by grouping the data by content_id and confirming there are no duplicates. Below is the actual DataFrame representing our unit of analysis along with its features and the engineered target column.

In [15]:
# Construct our unit of analysis dataframe with features and our target
unit_of_analysis_df = df[[
    'content_id',
    'client_id',
    'impressions_90d',
    'avg_position',
    'ctr',
    'trend_direction'
]].copy()

# Safely engineer the target label without leakage
unit_of_analysis_df['is_declining_label'] = (unit_of_analysis_df['trend_direction'] == 'down').astype(int)

# Verify unit of analysis grain (expect empty output when checking duplicates)
duplicates = unit_of_analysis_df.groupby('content_id').filter(lambda x: len(x) > 1)
print(f"Duplicate rows at the content_id grain: {len(duplicates)}")

# Display actual dataframe structure
unit_of_analysis_df.head(5)


Duplicate rows at the content_id grain: 0


,content_id,client_id,impressions_90d,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

While simple heuristics like "days since last update > 180" are easy to write, they fail to scale under real-world complexity:

Multi-Dimensionality: A page decays due to complex, intersecting factors, such as a dropping average position combined with a dropping click-through rate, despite impressions remaining steady. Manual rules cannot dynamically balance these features.

Unbalanced Client Profiles: Our inventory features high-performing evergreen articles that are old but highly active, alongside newer pages that require pruning. A flat threshold would misclassify both, wasting precious editorial hours. An ML model learns how these signals interact to prioritize work based on true opportunity.

In [16]:
# Proof of rule complexity: Let's show how many old pages (age >= 180 days) are actually
# NOT declining (which a static age-based rule would falsely flag for a refresh, wasting budget)
unnecessary_audits = len(df[(df['content_age_days'] >= 180) & (df['trend_direction'] != 'down')])
pct_waste = (unnecessary_audits / len(df[df['content_age_days'] >= 180])) * 100

print("--- HEURISTIC INEFFICIENCY PROOF ---")
print(f"Old pages that are actually healthy (False Positives): {unnecessary_audits:,}")
print(f"Waste rate of a flat age-based heuristic rule: {pct_waste:.1f}%")


--- HEURISTIC INEFFICIENCY PROOF ---
Old pages that are actually healthy (False Positives): 9,247
Waste rate of a flat age-based heuristic rule: 51.4%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.